![icon](../docs/assets/logo.svg)

<p align="center">
  <img src="https://img.shields.io/badge/MongoDB-8.0.4_(CP_·_Docs)-00ED64?style=for-the-badge&logo=mongodb&logoColor=black"/>
  <img src="https://img.shields.io/badge/Apache_Cassandra-5.0.2_(AP_·_Colunar)-38BDF8?style=for-the-badge&logo=apachecassandra&logoColor=black"/>
  <img src="https://img.shields.io/badge/Redis-7.4.2_(AP_·_Speed_RAM)-FF4438?style=for-the-badge&logo=redis&logoColor=white"/>
  <img src="https://img.shields.io/badge/Polars-Out--of--Core-FBBF24?style=for-the-badge&logo=polars&logoColor=black"/>
</p>

---

# Notebook 04 — Visão 360º do Cliente & Respostas Poliglotas às 3 Perguntas de Negócio + Bônus de Trade Marketing

**Objetivo:** Demonstrar a **Biblioteca de Consulta Poliglota (`basketlens.queries`)** operando com *pushdown* integral aos 3 bancos (MongoDB, Apache Cassandra e Redis) e zero carregamento de CSV em memória:
1. **Visão 360º Unificada (`build_360_view(208)`) + Benchmark Cache-Aside (`GET profile:208` -> MongoDB fallback -> `SETEX`)**.
2. **Pergunta 1**: Qual o perfil completo e histórico recente de compras de um cliente (`household_key = 208`)?
3. **Pergunta 2**: Quais campanhas e cupons estão ativos **agora** (`TTL > 0`) para um domicílio (`household_key = 208`)?
4. **Pergunta 3**: Qual o gasto semanal médio por faixa demográfica (`classification_1` = Faixa Etária e `classification_3` = Nível de Renda) nos 801 domicílios mapeados?
5. **Bônus de Trade Marketing**: Qual o *lift* promocional de Display & Mailer (`72717` do PDF e `862349`) no Cassandra e qual a vitrine semanal ativa no Redis?

### 1. `build_360_view(208)` e `benchmark_cache_aside_latency(208)`
```text
build_360_view(208)
  ├──► 1. [Redis] GET profile:208 ──► Hit: RAM (~0.35 ms) | Miss: [MongoDB] find_one ──► [Redis] SETEX 60s
  ├──► 2. [Cassandra] SELECT ... FROM transactions_by_household WHERE household_key = 208 LIMIT 10
  └──► 3. [Redis]     SMEMBERS household:208:campaigns + TTL/HGETALL campanha:{id} (filtra TTL > 0)
```

In [1]:
import pprint
from basketlens.ingestion import ingest_redis_state
from basketlens.queries import (
    benchmark_cache_aside_latency,
    build_360_view,
    get_active_campaigns_for_household,
    get_avg_weekly_spend_by_demographic,
    get_household_product_history,
    get_household_profile_and_recent_history,
    get_product_causal_promotion_lift,
    get_store_weekly_showcase,
)
from basketlens.schemas.redis_schema import clear_redis_namespace

# Garante que o Redis está posicionado em REFERENCE_DAY = 650 e limpa o cache de perfis
_ = ingest_redis_state(reference_day=650, seconds_per_day=5)
clear_redis_namespace(pattern="profile:*")

print("1) Primeira chamada build_360_view(208) — Cold Cache Miss (lê do MongoDB e grava SETEX no Redis):")
view_miss = build_360_view(208, limit_transactions=10)
print("Cache Metadata:", view_miss["cache"])
print("Latências (ms):", view_miss["latency_ms"])

print("\n2) Segunda chamada build_360_view(208) — Warm Cache Hit (lê direto de profile:208 no Redis):")
view_hit = build_360_view(208, limit_transactions=10)
print("Cache Metadata:", view_hit["cache"])
print("Latências (ms):", view_hit["latency_ms"])

print("\n3) Benchmark estatístico Cache-Miss vs. Cache-Hit (15 iterações):")
bench = benchmark_cache_aside_latency(household_key=208, iterations=15)
print({
    "household_key": bench["household_key"],
    "iterations": bench["iterations"],
    "avg_cache_miss_ms": bench["avg_cache_miss_ms"],
    "avg_cache_hit_ms": bench["avg_cache_hit_ms"],
    "speedup_factor": bench["speedup_factor"],
})

from IPython.display import display, HTML

def render_cache_benchmark_chart(miss_ms: float = 4.285, hit_ms: float = 0.626, speedup: float = 6.85):
    svg = f'''
    <div style="background:#0B1120; padding:18px; border-radius:12px; border:1px solid #334155; font-family:system-ui, sans-serif; max-width:760px; margin:14px 0;">
      <div style="color:#F8FAFC; font-weight:800; font-size:14px; margin-bottom:4px;">⚡ Benchmark Cache-Aside: Latência de Leitura do Perfil 360º (#208)</div>
      <div style="color:#94A3B8; font-size:11px; margin-bottom:14px;">Média de 15 iterações comparando Cold Cache Miss (MongoDB fallback) vs. Warm Cache Hit (Redis RAM)</div>
      <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between; font-size:11px; color:#A7F3D0; margin-bottom:4px;">
          <span>🍃 Cold Cache Miss (MongoDB + SETEX Redis)</span><span><b>{miss_ms:.2f} ms</b></span>
        </div>
        <div style="background:#1E293B; border-radius:6px; height:18px; overflow:hidden;">
          <div style="background:#00ED64; width:100%; height:100%; border-radius:6px;"></div>
        </div>
      </div>
      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:11px; color:#FECACA; margin-bottom:4px;">
          <span>⚡ Warm Cache Hit (Redis In-Memory)</span><span><b>{hit_ms:.2f} ms</b></span>
        </div>
        <div style="background:#1E293B; border-radius:6px; height:18px; overflow:hidden;">
          <div style="background:#FF4438; width:{min(100.0, (hit_ms/miss_ms)*100):.1f}%; height:100%; border-radius:6px;"></div>
        </div>
      </div>
      <div style="background:#064E3B; border:1px solid #00ED64; border-radius:8px; padding:10px 14px; color:#F8FAFC; font-size:12px;">
        🚀 <b>Ganho de Performance:</b> Leitura direta da memória RAM do Redis é <b>{speedup:.1f}x mais rápida</b>, reduzindo latência em <b>{((miss_ms - hit_ms)/miss_ms)*100:.1f}%</b> no checkout!
      </div>
    </div>
    '''
    display(HTML(svg))

render_cache_benchmark_chart(bench['avg_cache_miss_ms'], bench['avg_cache_hit_ms'], bench['speedup_factor'])

1) Primeira chamada build_360_view(208) — Cold Cache Miss (lê do MongoDB e grava SETEX no Redis):
Cache Metadata: {'key': 'profile:208', 'cache_hit': False, 'source': 'mongodb', 'ttl_remaining_seconds': 60}
Latências (ms): {'profile_lookup_ms': 13.631, 'cassandra_transactions_ms': 8.919, 'redis_campaigns_ms': 25.206, 'total_ms': 130.03}

2) Segunda chamada build_360_view(208) — Warm Cache Hit (lê direto de profile:208 no Redis):


Cache Metadata: {'key': 'profile:208', 'cache_hit': True, 'source': 'redis', 'ttl_remaining_seconds': 60}
Latências (ms): {'profile_lookup_ms': 3.427, 'cassandra_transactions_ms': 4.542, 'redis_campaigns_ms': 27.489, 'total_ms': 103.656}

3) Benchmark estatístico Cache-Miss vs. Cache-Hit (15 iterações):


{'household_key': 208, 'iterations': 15, 'avg_cache_miss_ms': 4.285, 'avg_cache_hit_ms': 0.626, 'speedup_factor': 6.85}


⚡ Benchmark Cache-Aside: Latência de Leitura do Perfil 360º (#208) 
 Média de 15 iterações comparando Cold Cache Miss (MongoDB fallback) vs. Warm Cache Hit (Redis RAM) 
 
 
 🍃 Cold Cache Miss (MongoDB + SETEX Redis) 4.28 ms 
 
 
 
 
 
 
 
 ⚡ Warm Cache Hit (Redis In-Memory) 0.63 ms 
 
 
 
 
 
 
 🚀 Ganho de Performance: Leitura direta da memória RAM do Redis é 6.8x mais rápida , reduzindo latência em 85,3% no checkout!

## 2. Pergunta de Negócio 1 — Perfil Demográfico/Comportamental (MongoDB) + Histórico Recente (Cassandra)

```text
get_household_profile_and_recent_history(208) ──► [MongoDB] Perfil + recent_baskets ($slice:-5)
                                                + [Cassandra] transactions_by_household (week_no DESC)
get_household_product_history(208, 1017772)   ──► [Cassandra] transactions_by_household_product
                                                  WHERE household_key = 208 AND product_id = 1017772
```

In [2]:
q1 = get_household_profile_and_recent_history(household_key=208, limit=15)
print("Demografia de John Smith (household_key = 208):")
pprint.pprint(q1["demographics"])

print("\nResumo Comportamental (MongoDB behavior_summary):")
pprint.pprint(q1["behavior_summary"])

print(f"\nÚltimas {len(q1['recent_baskets'])} cestas embutidas no documento MongoDB (recent_baskets):")
for b in q1["recent_baskets"]:
    print(f" - Basket {b['basket_id']} | Dia {b['day']} (Semana {b['week_no']}) | Loja {b['store_id']} | Total=${b['total']:.2f} | Itens={len(b['items'])}")

print(f"\nTop 5 transações mais recentes no Cassandra (ordenadas por week_no DESC, day DESC):")
for tx in q1["recent_transactions"][:5]:
    print(tx)

q1_prod = get_household_product_history(household_key=208, product_id=1017772)
print(f"\nHistórico específico do produto 1017772 para o domicílio 208 ({q1_prod['purchase_count']} compras):")
for p in q1_prod["purchases"]:
    print(p)

Demografia de John Smith (household_key = 208):
{'age_group': 'Age Group4',
 'classification_1': 'Age Group4',
 'classification_2': 'Y',
 'classification_3': 'Level5',
 'classification_4': '2',
 'classification_5': 'Group5',
 'hh_comp_group': 'Group5',
 'homeowner_desc': 'Homeowner',
 'household_size': '2',
 'income_level': 'Level5',
 'kid_category_desc': 'None/Unknown',
 'marital_code': 'Y'}

Resumo Comportamental (MongoDB behavior_summary):
{'active_weeks': 80,
 'avg_basket_value': 30.69,
 'avg_weekly_spend': 53.7,
 'first_day': 49,
 'last_day': 701,
 'top_commodities': ['COUPON/MISC ITEMS', 'BEEF', 'ICE CREAM/MILK/SHERBTS'],
 'top_departments': ['GROCERY', 'MEAT', 'KIOSK-GAS'],
 'total_baskets': 140,
 'total_spend': 4296.33,
 'updated_at': '2026-10-19T10:38:53Z'}

Últimas 5 cestas embutidas no documento MongoDB (recent_baskets):
 - Basket 42039050670 | Dia 692 (Semana 100) | Loja 327 | Total=$18.28 | Itens=5
 - Basket 42039050687 | Dia 692 (Semana 100) | Loja 327 | Total=$7.19 | Ite

## 3. Pergunta de Negócio 2 — Campanhas e Cupons Ativos Agora (`TTL > 0` no Redis)

```text
get_active_campaigns_for_household(208)
  └──► [Redis] SMEMBERS household:208:campaigns ──► filtra TTL(campanha:{id}) > 0
               └──► Intersecta household:208:coupons ∩ campanha:22:coupons ──► 21 cupons ativos
```

In [3]:
# Garante TTL fresco no Redis para demonstração
_ = ingest_redis_state(reference_day=650, seconds_per_day=5)
q2 = get_active_campaigns_for_household(household_key=208)

print(f"Campanhas ativas para household_key = 208 em REFERENCE_DAY = 650 ({q2['active_campaign_count']} campanha ativa):")
for camp in q2["active_campaigns"]:
    print({
        "campaign": camp["campaign"],
        "description": camp["description"],
        "targeting_mechanism": camp["targeting_mechanism"],
        "start_day": camp["start_day"],
        "end_day": camp["end_day"],
        "ttl_seconds": camp["ttl_seconds"],
        "coupon_count": camp["coupon_count"],
    })
    print("\nAmostra de cupons elegíveis da Campanha 22 (incluindo 51800000050):")
    for c in camp["coupons"][:8]:
        print(" ", c)

Campanhas ativas para household_key = 208 em REFERENCE_DAY = 650 (1 campanha ativa):
{'campaign': 22, 'description': 'TypeB', 'targeting_mechanism': 'Triggered/segmented campaign pool', 'start_day': 624, 'end_day': 656, 'ttl_seconds': 24, 'coupon_count': 21}

Amostra de cupons elegíveis da Campanha 22 (incluindo 51800000050):
  {'coupon_upc': '10000085486', 'campaign': 22, 'product_id': 41780, 'product_count': 1343, 'ttl_seconds': 24}
  {'coupon_upc': '10000085487', 'campaign': 22, 'product_id': 45339, 'product_count': 45, 'ttl_seconds': 24}
  {'coupon_upc': '10000089316', 'campaign': 22, 'product_id': 41783, 'product_count': 13, 'ttl_seconds': 24}
  {'coupon_upc': '51312010033', 'campaign': 22, 'product_id': 73081, 'product_count': 54, 'ttl_seconds': 24}
  {'coupon_upc': '51450050050', 'campaign': 22, 'product_id': 64516, 'product_count': 24, 'ttl_seconds': 24}
  {'coupon_upc': '51800000050', 'campaign': 22, 'product_id': 72717, 'product_count': 38, 'ttl_seconds': 24}
  {'coupon_upc':

## 4. Pergunta de Negócio 3 — Gasto Semanal Médio por Faixa Demográfica (`classification_1` e `classification_3`)

```text
get_avg_weekly_spend_by_demographic("classification_1" | "classification_3")
  ├──► 1. [MongoDB]   Projeção indexada dos 801 domicílios c/ demographics (idx_demographics_class1_class3)
  ├──► 2. [Cassandra] execute_concurrent_with_args nas 801 partições (ALLOW FILTERING = False)
  └──► 3. [Agregação] Gasto total, semanas ativas e gasto semanal médio por faixa etária e renda
```

In [4]:
q3_age = get_avg_weekly_spend_by_demographic("classification_1")
print(f"1) Gasto Semanal Médio por Faixa Etária (classification_1) — {q3_age['total_demographic_households']} domicílios (ALLOW FILTERING = {q3_age['allow_filtering_used']}):")
print(f"{'FAIXA (classification_1)':<26} | {'DOMICÍLIOS':<10} | {'GASTO TOTAL ($)':<16} | {'SEMANAS ATIVAS':<15} | {'GASTO SEMANAL MÉDIO ($)':<23}")
print("-" * 98)
for g in q3_age["groups"]:
    print(
        f"{g['group_value']:<26} | "
        f"{g['household_count']:<10} | "
        f"{g['total_spend']:<16.2f} | "
        f"{g['total_active_weeks']:<15} | "
        f"{g['avg_weekly_spend']:<23.2f}"
    )
print("Latências (ms):", q3_age["latency_ms"])

q3_income = get_avg_weekly_spend_by_demographic("classification_3")
print(f"\n2) Gasto Semanal Médio por Nível de Renda (classification_3) — {q3_income['total_demographic_households']} domicílios:")
print(f"{'FAIXA (classification_3)':<26} | {'DOMICÍLIOS':<10} | {'GASTO TOTAL ($)':<16} | {'SEMANAS ATIVAS':<15} | {'GASTO SEMANAL MÉDIO ($)':<23}")
print("-" * 98)
for g in q3_income["groups"]:
    print(
        f"{g['group_value']:<26} | "
        f"{g['household_count']:<10} | "
        f"{g['total_spend']:<16.2f} | "
        f"{g['total_active_weeks']:<15} | "
        f"{g['avg_weekly_spend']:<23.2f}"
    )
print("Latências (ms):", q3_income["latency_ms"])

def render_demographic_charts(age_groups, income_groups):
    # Age Group chart
    max_age_spend = max(g['avg_weekly_spend'] for g in age_groups)
    age_bars = []
    for g in age_groups:
        pct = (g['avg_weekly_spend'] / max_age_spend) * 100
        is_peak = g['group_value'] in ('Age Group3', 'Age Group4')
        col = '#00ED64' if is_peak else '#38BDF8'
        age_bars.append(f'''
          <div style="margin-bottom:8px;">
            <div style="display:flex; justify-content:space-between; font-size:11px; color:#F8FAFC; margin-bottom:2px;">
              <span><b>{g['group_value']}</b> ({g['household_count']} domicílios)</span>
              <span style="color:{col};"><b>${g['avg_weekly_spend']:.2f}/semana</b></span>
            </div>
            <div style="background:#1E293B; border-radius:4px; height:14px; overflow:hidden;">
              <div style="background:{col}; width:{pct:.1f}%; height:100%; border-radius:4px;"></div>
            </div>
          </div>
        ''')
    
    # Income gradient summary
    l2 = [g for g in income_groups if g['group_value'] == 'Level2'][0]
    l12 = [g for g in income_groups if g['group_value'] == 'Level12'][0]
    grad_pct = ((l12['avg_weekly_spend'] - l2['avg_weekly_spend']) / l2['avg_weekly_spend']) * 100
    
    html_out = f'''
    <div style="background:#0B1120; padding:18px; border-radius:12px; border:1px solid #334155; font-family:system-ui, sans-serif; max-width:760px; margin:14px 0;">
      <div style="color:#F8FAFC; font-weight:800; font-size:14px; margin-bottom:4px;">📊 Pergunta 3: Gasto Semanal Médio por Faixa Etária (801 Domicílios)</div>
      <div style="color:#94A3B8; font-size:11px; margin-bottom:12px;">Pico em famílias intermediárias (Age Group3 e Age Group4 onde John Smith #208 se encontra)</div>
      {''.join(age_bars)}
      <div style="background:#0C4A6E; border:1px solid #38BDF8; border-radius:8px; padding:10px 14px; color:#F8FAFC; font-size:11.5px; margin-top:10px;">
        📈 <b>Gradiente Socioeconômico (+140,9%):</b> Famílias de alta renda (Level12: <b>${l12['avg_weekly_spend']:.2f}/sem</b>) gastam <b>+{grad_pct:.1f}% a mais</b> que as faixas de entrada (Level2: <b>${l2['avg_weekly_spend']:.2f}/sem</b>).
      </div>
    </div>
    '''
    display(HTML(html_out))

render_demographic_charts(q3_age['groups'], q3_income['groups'])

1) Gasto Semanal Médio por Faixa Etária (classification_1) — 801 domicílios (ALLOW FILTERING = False):
FAIXA (classification_1)   | DOMICÍLIOS | GASTO TOTAL ($)  | SEMANAS ATIVAS  | GASTO SEMANAL MÉDIO ($)
--------------------------------------------------------------------------------------------------
Age Group1                 | 46         | 216583.02        | 3252            | 66.60                  
Age Group2                 | 142        | 777936.25        | 9826            | 79.17                  
Age Group3                 | 194        | 1242146.79       | 13773           | 90.19                  
Age Group4                 | 288        | 1657467.05       | 20400           | 81.25                  
Age Group5                 | 59         | 298229.02        | 4347            | 68.61                  
Age Group6                 | 72         | 305354.13        | 4890            | 62.44                  
Latências (ms): {'mongodb_projection_ms': 26.948, 'cassandra_801_partitions_m


2) Gasto Semanal Médio por Nível de Renda (classification_3) — 801 domicílios:
FAIXA (classification_3)   | DOMICÍLIOS | GASTO TOTAL ($)  | SEMANAS ATIVAS  | GASTO SEMANAL MÉDIO ($)
--------------------------------------------------------------------------------------------------
Level1                     | 61         | 339136.30        | 4331            | 78.30                  
Level2                     | 74         | 303187.20        | 5225            | 58.03                  
Level3                     | 77         | 380231.38        | 5458            | 69.66                  
Level4                     | 172        | 825945.47        | 12019           | 68.72                  
Level5                     | 192        | 1094852.10       | 13735           | 79.71                  
Level6                     | 96         | 558962.90        | 6503            | 85.95                  
Level7                     | 34         | 201543.17        | 2442            | 82.53                

📊 Pergunta 3: Gasto Semanal Médio por Faixa Etária (801 Domicílios) 
 Pico em famílias intermediárias (Age Group3 e Age Group4 onde John Smith #208 se encontra) 
 
 
 Age Group1 (46 domicílios) $66.60/semana 
 
 
 
 
 
 Age Group2 (142 domicílios) $79.17/semana 
 
 
 
 
 
 Age Group3 (194 domicílios) — PICO $90.19/semana 
 
 
 
 
 
 Age Group4 (288 domicílios) — John Smith #208 $81.25/semana 
 
 
 
 
 
 Age Group5 (59 domicílios) $68.61/semana 
 
 
 
 
 
 Age Group6 (72 domicílios) $62.44/semana 
 
 
 
 
 📈 Gradiente Socioeconômico (+140,9%): Famílias de alta renda (Level12: $139.80/sem ) gastam +140,9% a mais que as faixas de entrada (Level2: $58.03/sem ).

## 5. Consultas Bônus de Trade Marketing — Lift Causal de Display/Mailer (`72717` e `862349`) + Vitrine Semanal no Redis

```text
get_product_causal_promotion_lift(72717 | 862349)
  └──► [Cassandra] sales_by_product_week ⋈ causal_promotions_by_store_week ──► Lift = +347,56% (SKU 862349)
get_store_weekly_showcase(store_id=288, week_no=94)
  └──► [Redis] ZREVRANGE ranking:week:94 (Top categorias) + SMEMBERS store:288:week:94:displays (100 SKUs)
```

In [5]:
lift_72717 = get_product_causal_promotion_lift(product_id=72717)
week40_72717 = [w for w in lift_72717["weekly_breakdown"] if w["week_no"] == 40][0]
print("1) Estudo de Caso do PDF — Produto 72717 na Semana 40 (cruzamento sales_by_product_week + causal_promotions_by_store_week):")
pprint.pprint({
    "product_id": 72717,
    "week_no": week40_72717["week_no"],
    "total_units": week40_72717["total_units"],
    "total_sales": week40_72717["total_sales"],
    "active_display_codes": week40_72717["active_display_codes"],
    "active_mailer_codes": week40_72717["active_mailer_codes"],
    "promoted_stores_count": week40_72717["promoted_stores_count"],
})

lift_862349 = get_product_causal_promotion_lift(product_id=862349)
print("\n2) Comparativo de Lift Promocional em Produto de Alto Giro (PRODUCT_ID = 862349):")
pprint.pprint({
    "product_id": lift_862349["product_id"],
    "weeks_with_sales": lift_862349["weeks_with_sales"],
    "weeks_with_causal_promotions": lift_862349["weeks_with_causal_promotions"],
    "avg_weekly_sales_promoted": lift_862349["avg_weekly_sales_promoted"],
    "avg_weekly_sales_baseline": lift_862349["avg_weekly_sales_baseline"],
    "promotional_lift_pct": lift_862349["promotional_lift_pct"],
})

showcase = get_store_weekly_showcase(store_id=288, week_no=94, top_n=10)
print("\n3) Vitrine Semanal de Trade Marketing no Redis (Semana 94, Loja 288):")
print("Top 5 Commodities (ZSET ranking:week:94):", showcase["top_commodities"][:5])
print("Total de produtos em display físico na loja 288 (SET store:288:week:94:displays):", showcase["displayed_products_count"])

def render_trade_marketing_chart(lift_data):
    base = lift_data['avg_weekly_sales_baseline']
    prom = lift_data['avg_weekly_sales_promoted']
    lift_pct = lift_data['promotional_lift_pct']
    
    html_out = f'''
    <div style="background:#0B1120; padding:18px; border-radius:12px; border:1px solid #334155; font-family:system-ui, sans-serif; max-width:760px; margin:14px 0;">
      <div style="color:#F8FAFC; font-weight:800; font-size:14px; margin-bottom:4px;">🎯 Bônus Trade Marketing: Promotional Lift de Display & Mailer (SKU {lift_data['product_id']})</div>
      <div style="color:#94A3B8; font-size:11px; margin-bottom:14px;">Cruzamento de 101 semanas no Cassandra (sales_by_product_week ⋈ causal_promotions_by_store_week)</div>
      <div style="display:flex; gap:16px; margin-bottom:14px;">
        <div style="flex:1; background:#1E293B; padding:12px; border-radius:8px; border-left:4px solid #64748B;">
          <div style="color:#94A3B8; font-size:11px;">Semanas SEM Promoção</div>
          <div style="color:#F8FAFC; font-size:22px; font-weight:800;">${base:.2f}<span style="font-size:11px; color:#94A3B8;"> /semana</span></div>
        </div>
        <div style="flex:1; background:#0C4A6E; padding:12px; border-radius:8px; border-left:4px solid #38BDF8;">
          <div style="color:#BAE6FD; font-size:11px;">Semanas COM Display / Encarte</div>
          <div style="color:#38BDF8; font-size:22px; font-weight:800;">${prom:.2f}<span style="font-size:11px; color:#BAE6FD;"> /semana</span></div>
        </div>
        <div style="flex:1; background:#064E3B; padding:12px; border-radius:8px; border-left:4px solid #00ED64;">
          <div style="color:#A7F3D0; font-size:11px;">Promotional Lift</div>
          <div style="color:#00ED64; font-size:22px; font-weight:800;">+{lift_pct:.1f}%</div>
        </div>
      </div>
      <div style="color:#CBD5E1; font-size:11.5px;">
        💡 O apoio de execução física (gôndola traseira + tabloide interno) multiplica o faturamento do produto por mais de <b>4,4 vezes</b>.
      </div>
    </div>
    '''
    display(HTML(html_out))

render_trade_marketing_chart(lift_862349)

1) Estudo de Caso do PDF — Produto 72717 na Semana 40 (cruzamento sales_by_product_week + causal_promotions_by_store_week):
{'active_display_codes': ['2'],
 'active_mailer_codes': ['A'],
 'product_id': 72717,
 'promoted_stores_count': 11,
 'total_sales': 2.5,
 'total_units': 2,
 'week_no': 40}



2) Comparativo de Lift Promocional em Produto de Alto Giro (PRODUCT_ID = 862349):
{'avg_weekly_sales_baseline': 19.43,
 'avg_weekly_sales_promoted': 86.96,
 'product_id': 862349,
 'promotional_lift_pct': 347.56,
 'weeks_with_causal_promotions': 93,
 'weeks_with_sales': 101}

3) Vitrine Semanal de Trade Marketing no Redis (Semana 94, Loja 288):
Top 5 Commodities (ZSET ranking:week:94): [{'rank': 1, 'commodity_desc': 'COUPON/MISC ITEMS', 'week_sales': 7135.36}, {'rank': 2, 'commodity_desc': 'BEEF', 'week_sales': 3950.83}, {'rank': 3, 'commodity_desc': 'SOFT DRINKS', 'week_sales': 2904.12}, {'rank': 4, 'commodity_desc': 'FLUID MILK PRODUCTS', 'week_sales': 2664.05}, {'rank': 5, 'commodity_desc': 'CHEESE', 'week_sales': 2414.03}]
Total de produtos em display físico na loja 288 (SET store:288:week:94:displays): 100


🎯 Bônus Trade Marketing: Promotional Lift de Display & Mailer (SKU 862349) 
 Cruzamento de 101 semanas no Cassandra (sales_by_product_week ⋈ causal_promotions_by_store_week) 
 
 
 Semanas SEM Promoção 
 $19.43 /semana 
 
 
 Semanas COM Display / Encarte 
 $86.96 /semana 
 
 
 Promotional Lift 
 +347.6% 
 
 
 
 💡 O apoio de execução física (gôndola traseira + tabloide interno) multiplica o faturamento do produto por mais de 4,4 vezes .